# S23DR 2026 — 3-D Data Visualisation

Interactive Plotly charts for every signal in the dataset:

1. Setup
2. Load a sample
3. Point cloud — coloured by **height (Z)**
4. Point cloud — coloured by **semantic class** (`class_id`)
5. Point cloud — coloured by **view-agreement** (`vote_frac`)
6. Point cloud — coloured by **view count** (`n_views_voted`)
7. Point cloud — coloured by **mask**
8. GT wireframe overlay
9. Multi-sample grid (N random houses side-by-side)
10. Dataset statistics (vertex/edge counts, class distributions)

## 1 · Setup

In [ ]:
!pip install -q datasets huggingface_hub plotly numpy

In [ ]:
import os

REPO_DIR = "/content/3d_building_construction"

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull origin main
else:
    !git clone https://github.com/12turtleships/3d_building_construction {REPO_DIR}

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

## 2 · Load samples

Streams a few rows from the public validation split — no local download needed.

In [ ]:
# ── configuration ─────────────────────────────────────────────────────────────
HF_DATASET  = "usm3d/s23dr-2026-sampled_4096_v2"
SPLIT       = "validation"
N_LOAD      = 20       # number of samples to load (more = slower)
MAX_PTS     = 4096     # max points per cloud to render (reduce if sluggish)
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
import io, zipfile
import numpy as np
from datasets import load_dataset


def unpack(row):
    out = {}
    with zipfile.ZipFile(io.BytesIO(row["data"])) as zf:
        for name in zf.namelist():
            if name.endswith(".npy"):
                out[name[:-4]] = np.load(io.BytesIO(zf.read(name)), allow_pickle=False)
    out["order_id"] = row.get("order_id", "")
    return out


raw = list(load_dataset(HF_DATASET, split=SPLIT, streaming=False).select(range(N_LOAD)))
samples = [unpack(r) for r in raw]

print(f"Loaded {len(samples)} samples")
print()
print("Arrays in sample 0:")
s0 = samples[0]
print(f"  {'Array':22s}  {'Shape':18s}  {'Dtype':10s}  {'Min':>8s}  {'Max':>8s}")
print("  " + "-"*72)
for k, v in sorted(s0.items()):
    if isinstance(v, np.ndarray):
        print(f"  {k:22s}  {str(v.shape):18s}  {str(v.dtype):10s}  "
              f"{float(v.min()):>8.3f}  {float(v.max()):>8.3f}")

## 3 · Point cloud — coloured by height (Z)

The classic first look: Z-axis gives an immediate sense of roof geometry.

In [ ]:
import plotly.graph_objects as go

SAMPLE_IDX = 0   # ← change to inspect a different sample

s = samples[SAMPLE_IDX]
xyz = s["xyz_norm"]
rng = np.random.default_rng(42)
idx = rng.choice(len(xyz), min(MAX_PTS, len(xyz)), replace=False)
pc  = xyz[idx]

fig = go.Figure(go.Scatter3d(
    x=pc[:, 0], y=pc[:, 1], z=pc[:, 2],
    mode="markers",
    marker=dict(
        size=1.8,
        color=pc[:, 2],
        colorscale="Viridis",
        colorbar=dict(title="Z (norm)", thickness=12),
        opacity=0.8,
    ),
    name="point cloud",
))
fig.update_layout(
    title=f"{s['order_id']}  —  coloured by height (Z)",
    scene=dict(aspectmode="data",
               xaxis_title="X", yaxis_title="Y", zaxis_title="Z"),
    margin=dict(l=0, r=0, t=40, b=0), height=600,
)
fig.show()
print(f"{len(pc):,} points rendered")

## 4 · Point cloud — coloured by semantic class (`class_id`)

`class_id` encodes the surface type each point was voted onto during SfM
(e.g. roof, wall, window, sky).  Each unique ID gets a distinct colour.

In [ ]:
s   = samples[SAMPLE_IDX]
xyz = s["xyz_norm"]
cid = s["class_id"]

idx = rng.choice(len(xyz), min(MAX_PTS, len(xyz)), replace=False)
pc  = xyz[idx]
cc  = cid[idx]

unique_classes = np.unique(cc)
print(f"Unique class IDs in this sample: {unique_classes.tolist()}")

# Assign a colour index 0..N to each unique class for a qualitative palette
class_to_idx = {c: i for i, c in enumerate(unique_classes)}
colour_val   = np.array([class_to_idx[c] for c in cc], dtype=float)

fig = go.Figure(go.Scatter3d(
    x=pc[:, 0], y=pc[:, 1], z=pc[:, 2],
    mode="markers",
    marker=dict(
        size=1.8,
        color=colour_val,
        colorscale="Rainbow",
        colorbar=dict(title="class_id", thickness=12,
                      tickvals=list(class_to_idx.values()),
                      ticktext=[str(c) for c in class_to_idx]),
        opacity=0.85,
    ),
))
fig.update_layout(
    title=f"{s['order_id']}  —  coloured by class_id",
    scene=dict(aspectmode="data",
               xaxis_title="X", yaxis_title="Y", zaxis_title="Z"),
    margin=dict(l=0, r=0, t=40, b=0), height=600,
)
fig.show()

## 5 · Point cloud — coloured by view-agreement (`vote_frac`)

`vote_frac` = fraction of SfM camera views that agree on a point's 3-D
position.  High values (→ yellow) = reliable; low values (→ purple) = noisy.

In [ ]:
s   = samples[SAMPLE_IDX]
xyz = s["xyz_norm"]
vf  = s["vote_frac"]

idx = rng.choice(len(xyz), min(MAX_PTS, len(xyz)), replace=False)
pc  = xyz[idx]
vc  = vf[idx]

fig = go.Figure(go.Scatter3d(
    x=pc[:, 0], y=pc[:, 1], z=pc[:, 2],
    mode="markers",
    marker=dict(
        size=1.8,
        color=vc,
        cmin=0.0, cmax=1.0,
        colorscale="Plasma",
        colorbar=dict(title="vote_frac", thickness=12),
        opacity=0.85,
    ),
))
fig.update_layout(
    title=f"{s['order_id']}  —  coloured by vote_frac (view agreement)",
    scene=dict(aspectmode="data",
               xaxis_title="X", yaxis_title="Y", zaxis_title="Z"),
    margin=dict(l=0, r=0, t=40, b=0), height=600,
)
fig.show()
print(f"vote_frac  mean={vc.mean():.3f}  std={vc.std():.3f}  "
      f"high (>0.8): {(vc > 0.8).mean()*100:.1f}%  "
      f"low  (<0.3): {(vc < 0.3).mean()*100:.1f}%")

## 6 · Point cloud — coloured by view count (`n_views_voted`)

In [ ]:
s   = samples[SAMPLE_IDX]
xyz = s["xyz_norm"]
nv  = s["n_views_voted"].astype(float)

idx = rng.choice(len(xyz), min(MAX_PTS, len(xyz)), replace=False)
pc  = xyz[idx]
nc  = nv[idx]

fig = go.Figure(go.Scatter3d(
    x=pc[:, 0], y=pc[:, 1], z=pc[:, 2],
    mode="markers",
    marker=dict(
        size=1.8,
        color=nc,
        colorscale="Turbo",
        colorbar=dict(title="n_views", thickness=12),
        opacity=0.85,
    ),
))
fig.update_layout(
    title=f"{s['order_id']}  —  coloured by n_views_voted",
    scene=dict(aspectmode="data",
               xaxis_title="X", yaxis_title="Y", zaxis_title="Z"),
    margin=dict(l=0, r=0, t=40, b=0), height=600,
)
fig.show()
print(f"n_views_voted  min={int(nc.min())}  max={int(nc.max())}  "
      f"mean={nc.mean():.1f}  median={np.median(nc):.1f}")

## 7 · Point cloud — coloured by mask

Valid points (mask=1, blue) vs filtered points (mask=0, red).

In [ ]:
s    = samples[SAMPLE_IDX]
xyz  = s["xyz_norm"]
mask = s["mask"].astype(bool)

idx = rng.choice(len(xyz), min(MAX_PTS, len(xyz)), replace=False)
pc  = xyz[idx]
mk  = mask[idx]

valid_trace = go.Scatter3d(
    x=pc[mk, 0], y=pc[mk, 1], z=pc[mk, 2],
    mode="markers",
    marker=dict(size=1.8, color="royalblue", opacity=0.7),
    name=f"valid  ({mk.sum():,})",
)
masked_trace = go.Scatter3d(
    x=pc[~mk, 0], y=pc[~mk, 1], z=pc[~mk, 2],
    mode="markers",
    marker=dict(size=2.5, color="tomato", opacity=0.9),
    name=f"masked ({(~mk).sum():,})",
)

fig = go.Figure([valid_trace, masked_trace])
fig.update_layout(
    title=f"{s['order_id']}  —  mask  (blue=valid, red=filtered)",
    scene=dict(aspectmode="data",
               xaxis_title="X", yaxis_title="Y", zaxis_title="Z"),
    legend=dict(x=0, y=1),
    margin=dict(l=0, r=0, t=40, b=0), height=600,
)
fig.show()
print(f"Valid: {mk.sum():,} / {len(mk):,}  ({mk.mean()*100:.1f}%)")

## 8 · GT wireframe overlay

Point cloud (blue) + ground-truth roof wireframe (green lines).

The wireframe is stored as `gt_segments` in **normalised space** — the same
space as `xyz_norm` — so they can be overlaid directly.

In [ ]:
s    = samples[SAMPLE_IDX]
xyz  = s["xyz_norm"]
segs = s["gt_segments"]   # (E, 2, 3) in normalised space

idx = rng.choice(len(xyz), min(MAX_PTS, len(xyz)), replace=False)
pc  = xyz[idx]

# Point cloud
pc_trace = go.Scatter3d(
    x=pc[:, 0], y=pc[:, 1], z=pc[:, 2],
    mode="markers",
    marker=dict(size=1.5, color="royalblue", opacity=0.3),
    name="point cloud",
)

# Wireframe segments → NaN-separated polyline
xs, ys, zs = [], [], []
for seg in segs:
    xs += [float(seg[0, 0]), float(seg[1, 0]), None]
    ys += [float(seg[0, 1]), float(seg[1, 1]), None]
    zs += [float(seg[0, 2]), float(seg[1, 2]), None]

wf_trace = go.Scatter3d(
    x=xs, y=ys, z=zs,
    mode="lines",
    line=dict(color="limegreen", width=4),
    name=f"GT wireframe ({len(segs)} segs)",
)

# Vertex markers
gt_v  = s.get("gt_vertices")   # world space — convert to normalised
scale = float(s.get("scale", 1.0))
center = s.get("center", np.zeros(3))
if gt_v is not None:
    gt_v_norm = (gt_v - center) / scale
    vert_trace = go.Scatter3d(
        x=gt_v_norm[:, 0], y=gt_v_norm[:, 1], z=gt_v_norm[:, 2],
        mode="markers",
        marker=dict(size=6, color="limegreen", symbol="diamond"),
        name=f"GT vertices ({len(gt_v_norm)})",
    )
    traces = [pc_trace, wf_trace, vert_trace]
else:
    traces = [pc_trace, wf_trace]

fig = go.Figure(traces)
fig.update_layout(
    title=f"{s['order_id']}  —  point cloud + GT wireframe",
    scene=dict(aspectmode="data",
               xaxis_title="X", yaxis_title="Y", zaxis_title="Z"),
    legend=dict(x=0, y=1),
    margin=dict(l=0, r=0, t=40, b=0), height=650,
)
fig.show()
print(f"GT segments : {len(segs)}")
if gt_v is not None:
    print(f"GT vertices : {len(gt_v)}")

## 9 · All signals in one figure

Four sub-charts for the same sample using Plotly subplots:
height · class_id · vote_frac · GT wireframe.

In [ ]:
from plotly.subplots import make_subplots

s    = samples[SAMPLE_IDX]
xyz  = s["xyz_norm"]
segs = s["gt_segments"]

idx = rng.choice(len(xyz), min(MAX_PTS, len(xyz)), replace=False)
pc  = xyz[idx]

def _pc_trace(color, cscale, ctitle, **kwargs):
    return go.Scatter3d(
        x=pc[:, 0], y=pc[:, 1], z=pc[:, 2],
        mode="markers",
        marker=dict(size=1.5, color=color, colorscale=cscale,
                    colorbar=dict(title=ctitle, thickness=10), opacity=0.8),
        **kwargs,
    )

def _seg_trace():
    xs, ys, zs = [], [], []
    for seg in segs:
        xs += [float(seg[0,0]), float(seg[1,0]), None]
        ys += [float(seg[0,1]), float(seg[1,1]), None]
        zs += [float(seg[0,2]), float(seg[1,2]), None]
    return go.Scatter3d(x=xs, y=ys, z=zs, mode="lines",
                        line=dict(color="limegreen", width=4))

cid   = s["class_id"][idx].astype(float)
vf    = s["vote_frac"][idx]

fig = make_subplots(
    rows=2, cols=2,
    specs=[[{"type": "scatter3d"}, {"type": "scatter3d"}],
           [{"type": "scatter3d"}, {"type": "scatter3d"}]],
    subplot_titles=("Height (Z)", "class_id", "vote_frac", "GT wireframe"),
)

fig.add_trace(_pc_trace(pc[:,2], "Viridis",  "Z"),     row=1, col=1)
fig.add_trace(_pc_trace(cid,     "Rainbow",  "class"), row=1, col=2)
fig.add_trace(_pc_trace(vf,      "Plasma",   "vf"),    row=2, col=1)
fig.add_trace(_pc_trace(pc[:,2], "Viridis",  "Z", opacity=0.2), row=2, col=2)
fig.add_trace(_seg_trace(),  row=2, col=2)

fig.update_layout(
    title_text=f"{s['order_id']} — all signals",
    height=900,
    showlegend=False,
    margin=dict(l=0, r=0, t=60, b=0),
)
for scene in ["scene", "scene2", "scene3", "scene4"]:
    fig.update_layout(**{scene: dict(aspectmode="data")})

fig.show()

## 10 · Multi-sample gallery

Grid of N houses, each showing the GT wireframe overlay.
Good for spotting variety in roof shapes.

In [ ]:
from plotly.subplots import make_subplots

GALLERY_N   = 6    # number of houses to show (keep ≤ 9 for readability)
GALLERY_PTS = 1024

cols = min(3, GALLERY_N)
rows = (GALLERY_N + cols - 1) // cols

fig = make_subplots(
    rows=rows, cols=cols,
    specs=[[{"type": "scatter3d"}] * cols for _ in range(rows)],
    subplot_titles=[samples[i]["order_id"] for i in range(GALLERY_N)],
)

rng2 = np.random.default_rng(0)

for n in range(GALLERY_N):
    s     = samples[n]
    xyz   = s["xyz_norm"]
    segs  = s["gt_segments"]
    row_i = n // cols + 1
    col_i = n %  cols + 1

    sel = rng2.choice(len(xyz), min(GALLERY_PTS, len(xyz)), replace=False)
    pc  = xyz[sel]

    # Point cloud
    fig.add_trace(go.Scatter3d(
        x=pc[:,0], y=pc[:,1], z=pc[:,2],
        mode="markers",
        marker=dict(size=1.2, color=pc[:,2], colorscale="Blues", opacity=0.4),
        showlegend=False,
    ), row=row_i, col=col_i)

    # Wireframe
    xs, ys, zs = [], [], []
    for seg in segs:
        xs += [float(seg[0,0]), float(seg[1,0]), None]
        ys += [float(seg[0,1]), float(seg[1,1]), None]
        zs += [float(seg[0,2]), float(seg[1,2]), None]
    fig.add_trace(go.Scatter3d(
        x=xs, y=ys, z=zs, mode="lines",
        line=dict(color="limegreen", width=3),
        showlegend=False,
    ), row=row_i, col=col_i)

scene_id = 1
for r in range(1, rows + 1):
    for c in range(1, cols + 1):
        key = "scene" if scene_id == 1 else f"scene{scene_id}"
        fig.update_layout(**{key: dict(aspectmode="data",
                                       xaxis=dict(visible=False),
                                       yaxis=dict(visible=False),
                                       zaxis=dict(visible=False))})
        scene_id += 1

fig.update_layout(
    title_text=f"Gallery: {GALLERY_N} houses from {SPLIT} split",
    height=380 * rows,
    margin=dict(l=0, r=0, t=60, b=0),
)
fig.show()

## 11 · Dataset statistics

Distribution histograms across all loaded samples:
- GT vertex count per house
- GT edge (segment) count per house
- `vote_frac` distribution
- `class_id` frequency

In [ ]:
import plotly.express as px
from plotly.subplots import make_subplots

n_verts = []
n_segs  = []
all_vf  = []
all_cid = []

for s in samples:
    gv = s.get("gt_vertices")
    gs = s.get("gt_segments")
    if gv is not None: n_verts.append(len(gv))
    if gs is not None: n_segs.append(len(gs))
    all_vf.extend(s["vote_frac"].tolist())
    all_cid.extend(s["class_id"].tolist())

all_vf  = np.array(all_vf)
all_cid = np.array(all_cid)

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=("GT vertex count", "GT edge (segment) count",
                    "vote_frac distribution", "class_id frequency"),
)

fig.add_trace(go.Histogram(x=n_verts, nbinsx=20, name="n_vertices",
                           marker_color="steelblue"), row=1, col=1)
fig.add_trace(go.Histogram(x=n_segs,  nbinsx=20, name="n_edges",
                           marker_color="seagreen"),  row=1, col=2)
fig.add_trace(go.Histogram(x=all_vf,  nbinsx=40, name="vote_frac",
                           marker_color="darkorange"), row=2, col=1)

# class_id bar chart
unique_cids, counts = np.unique(all_cid, return_counts=True)
fig.add_trace(go.Bar(x=unique_cids.tolist(), y=counts.tolist(),
                     name="class_id", marker_color="mediumpurple"), row=2, col=2)

fig.update_layout(
    title_text=f"Statistics across {len(samples)} samples",
    height=700, showlegend=False,
    margin=dict(l=40, r=20, t=60, b=40),
)
fig.update_xaxes(title_text="vertex count", row=1, col=1)
fig.update_xaxes(title_text="edge count",   row=1, col=2)
fig.update_xaxes(title_text="vote_frac",    row=2, col=1)
fig.update_xaxes(title_text="class_id",     row=2, col=2)
fig.show()

if n_verts:
    print(f"GT vertices / house  — min={min(n_verts)}  max={max(n_verts)}  "
          f"mean={np.mean(n_verts):.1f}  median={np.median(n_verts):.0f}")
if n_segs:
    print(f"GT segments / house  — min={min(n_segs)}  max={max(n_segs)}  "
          f"mean={np.mean(n_segs):.1f}  median={np.median(n_segs):.0f}")
print(f"vote_frac            — mean={all_vf.mean():.3f}  std={all_vf.std():.3f}")
print(f"class_id unique values: {len(unique_cids)}")

## 12 · Top-down (bird's-eye) view

XY projection — useful for comparing roof footprint shape to the wireframe.

In [ ]:
s    = samples[SAMPLE_IDX]
xyz  = s["xyz_norm"]
segs = s["gt_segments"]
vf   = s["vote_frac"]

idx = rng.choice(len(xyz), min(MAX_PTS, len(xyz)), replace=False)
pc  = xyz[idx]
vc  = vf[idx]

fig = go.Figure()

# Point cloud (XY)
fig.add_trace(go.Scatter(
    x=pc[:, 0], y=pc[:, 1],
    mode="markers",
    marker=dict(size=2, color=vc, colorscale="Plasma",
                colorbar=dict(title="vote_frac", thickness=12), opacity=0.6),
    name="point cloud (XY)",
))

# Wireframe (XY projection)
for seg in segs:
    fig.add_trace(go.Scatter(
        x=[seg[0, 0], seg[1, 0]],
        y=[seg[0, 1], seg[1, 1]],
        mode="lines",
        line=dict(color="limegreen", width=2),
        showlegend=False,
    ))

fig.update_layout(
    title=f"{s['order_id']}  —  top-down view (XY), wireframe overlay",
    xaxis_title="X (norm)", yaxis_title="Y (norm)",
    yaxis_scaleanchor="x",
    height=550,
    margin=dict(l=40, r=20, t=40, b=40),
    showlegend=False,
)
fig.show()